# Анализ новой логики определения дефолтности

## Предложенная бизнесом логика

Компания признаётся в состоянии дефолта, если у неё **одновременно** присутствуют два типа факторов:

| Сегмент | Триггер-фактор | Платёжный фактор (любой из) |
|---------|---------------|----------------------------|
| **ККБ** (ДКБ) | 13 | 15.2 \| 15.2.3 \| 15.2.2 |
| **МСБ** | 13 | 15.1.2 \| 15.2У |
| **МкБ** | 14 | 15.1.1 \| 15.2.2.1 |

**Суть:** триггер-фактор отражает ухудшение качества заёмщика (сигнальный/критический), а платёжный фактор — наличие просроченной задолженности. Сочетание обоих считается признаком дефолта.

---

## Открытые вопросы для бизнеса

Перед полноценной реализацией логики рекомендуется уточнить у коллег следующее:

### Временные рамки
1. **Максимальное окно между факторами.** Если ФП 13 возник в январе 2023, а ФП 15.2 — в декабре 2025, считается ли это дефолтом? Нужно ли ограничить окно (например, 6 / 12 / 18 месяцев)?
2. **Важен ли порядок появления?** Должен ли триггер-фактор (13/14) появиться **раньше** платёжного (15.x), или порядок не имеет значения?
3. **Одновременность.** Должны ли оба фактора быть **активны одновременно**, или достаточно, чтобы они оба появились у компании хотя бы раз за период?

### Логика учёта
4. **Повторные возникновения.** Если ФП 13 появился, был снят, а потом возник повторно — какую дату считать? Первую? Последнюю? Каждый случай отдельно?
5. **Учёт TYPE_FP.** Факторы 13, 14, 15.x — это ФП, СФП или и то и другое? Нужно ли учитывать тип?
6. **Что считать «снятием» фактора?** Есть ли в данных информация о закрытии/деактивации ФП?

### Сегментация
7. **Мигрирующие компании.** Если ИНН присутствует в нескольких сегментах (МкБ → МСБ), по какому сегменту проверять? По текущему? По тому, где возник триггер?
8. **Только ФП из своего сегмента?** ФП 13 из записи с X_AREA_RESP = ДКБ проверяется только для ККБ-логики, или если у компании есть ФП 13 из любого сегмента?

### Бизнес-контекст
9. **Цель использования.** Для чего будет использоваться результат — для ретроспективного анализа, для расчёта PD, для формирования резервов?
10. **Сравнение с текущей логикой.** Существует ли текущая (действующая) логика определения дефолта, с которой нужно сравнить результаты?
11. **Достаточность двух факторов.** Рассматривались ли более сложные комбинации (3+ фактора)? Почему выбраны именно эти пары?
12. **Пороговая просрочка.** Платёжные факторы (15.x) имеют разные пороги (30, 60, 90 дней). Влияет ли конкретный порог на решение о дефолте?

## 0. Импорт и параметры

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

# ── Пути к файлам ──
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

# ── Период анализа ──
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

# ── Маппинг X_AREA_RESP → сегмент ──
SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

# ══════════════════════════════════════════════════════════════════════════
# ЛОГИКА ДЕФОЛТА — конфигурация, предложенная бизнесом
# ══════════════════════════════════════════════════════════════════════════
# Для каждого сегмента определяем:
#   trigger  — множество «триггер-факторов» (ухудшение качества заёмщика)
#   payment  — множество «платёжных факторов» (просроченная задолженность)
# Компания в дефолте, если у неё есть хотя бы один фактор из trigger
# И хотя бы один фактор из payment.
# ══════════════════════════════════════════════════════════════════════════
DEFAULT_RULES = {
    "ККБ": {
        "trigger": {"13"},
        "payment": {"15.2", "15.2.3", "15.2.2"},
    },
    "МСБ": {
        "trigger": {"13"},
        "payment": {"15.1.2", "15.2У"},
    },
    "МкБ": {
        "trigger": {"14"},
        "payment": {"15.1.1", "15.2.2.1"},
    },
}


def normalize_inn(s):
    """Приводит ИНН к единому строковому формату (10 или 12 знаков)."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


print("Параметры и правила дефолта заданы.")
for seg, rules in DEFAULT_RULES.items():
    print(f"  {seg}: триггер {rules['trigger']}  +  платёж {rules['payment']}")

## 1. Загрузка и подготовка данных

In [ ]:
# Загрузка CRM
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
print(f"Загружено: {len(df):,} строк")

# Нормализация ИНН
df["inn"] = df["X_INN"].apply(normalize_inn)

# Преобразование даты идентификации
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

# Фильтр по периоду
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(df):,} строк")

# Переименование TYPE_FP
df["TYPE_FP"] = df["TYPE_FP"].replace({"FP": "ФП", "SFP": "СФП"})

# Маппинг сегментов
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

# Дедупликация
before = len(df)
df = df.drop_duplicates(subset=["inn", "NUMBER_FP_SFP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации: {len(df):,} строк (удалено {before - len(df):,} дублей)")

# Нормализация номера фактора (убираем пробелы по краям)
df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

print(f"\nУникальных ИНН: {df['inn'].nunique():,}")
print(f"Уникальных номеров факторов: {df['fp_num'].nunique():,}")

## 2. Проверка наличия факторов из правил в данных

Прежде чем применять логику, убедимся, что все указанные бизнесом номера факторов **реально встречаются** в выгрузке CRM. Если какого-то фактора нет — логика может быть неприменима.

In [ ]:
# Собираем все уникальные номера факторов, встречающиеся в данных
all_fp_in_data = set(df["fp_num"].dropna().unique())

print("Проверка наличия факторов из правил дефолта в данных CRM:\n")
print(f"{'Сегмент':<8} {'Роль':<10} {'Фактор':<12} {'Есть?':<8} {'Кол-во записей'}")
print("=" * 60)

missing_factors = []  # факторы, которых нет в данных

for seg in ["ККБ", "МСБ", "МкБ"]:
    rules = DEFAULT_RULES[seg]
    for role, factors in [("триггер", rules["trigger"]), ("платёж", rules["payment"])]:
        for fp in sorted(factors):
            # Считаем записи с этим фактором В РАМКАХ ЭТОГО СЕГМЕНТА
            cnt = len(df[(df["segment"] == seg) & (df["fp_num"] == fp)])
            exists = "Да" if cnt > 0 else "НЕТ ❌"
            print(f"{seg:<8} {role:<10} {fp:<12} {exists:<8} {cnt:,}")
            if cnt == 0:
                missing_factors.append((seg, role, fp))

if missing_factors:
    print(f"\n⚠️  ВНИМАНИЕ: {len(missing_factors)} фактор(ов) из правил НЕ НАЙДЕНО в данных!")
    for seg, role, fp in missing_factors:
        print(f"   {seg} / {role} / {fp}")
    print("\n   Это может означать:")
    print("   - Номер фактора записан иначе в CRM (проверьте регистр, пробелы, точки)")
    print("   - Фактор не возникал за анализируемый период")
    print("   - Фактор относится к другому сегменту")
else:
    print("\n✅ Все факторы из правил найдены в данных.")

### 2.1 Поиск похожих факторов (нечёткое совпадение)

Если какой-то фактор не найден, поищем похожие номера — возможно, отличается формат записи (например, `15.2` vs `15.2.` или `15.2 `).

In [ ]:
# Собираем все номера факторов, упомянутые в правилах
rule_factors = set()
for rules in DEFAULT_RULES.values():
    rule_factors |= rules["trigger"]
    rule_factors |= rules["payment"]

print("Поиск похожих номеров факторов в данных:\n")
for target in sorted(rule_factors):
    # Ищем факторы, которые начинаются с того же корня или содержат подстроку
    similar = sorted([f for f in all_fp_in_data
                      if f.startswith(target) or target.startswith(f)
                      or f.replace(".", "") == target.replace(".", "")])
    in_data = target in all_fp_in_data
    status = "✅" if in_data else "❌"
    print(f"{status} {target:<12}  Похожие в данных: {similar if similar else '—'}")

## 3. Применение логики дефолта

Для каждого ИНН в каждом сегменте проверяем:
1. Есть ли у него хотя бы один **триггер-фактор**
2. Есть ли у него хотя бы один **платёжный фактор**
3. Если оба условия выполнены → компания признаётся **дефолтной**

In [ ]:
results = {}  # {сегмент: DataFrame с результатами}

print("Применение логики дефолта (без ограничения по временному окну):\n")
print(f"{'Сегмент':<8} {'Уник. ИНН':<12} {'С триггером':<14} {'С платежом':<13} {'Дефолт':<10} {'% дефолта'}")
print("=" * 70)

for seg in ["МкБ", "МСБ", "ККБ"]:
    rules = DEFAULT_RULES[seg]
    seg_df = df[df["segment"] == seg].copy()
    
    # Множество ИНН, у которых есть триггер-фактор
    has_trigger = set(seg_df[seg_df["fp_num"].isin(rules["trigger"])]["inn"].unique())
    
    # Множество ИНН, у которых есть платёжный фактор
    has_payment = set(seg_df[seg_df["fp_num"].isin(rules["payment"])]["inn"].unique())
    
    # Дефолт = пересечение двух множеств
    default_inns = has_trigger & has_payment
    
    total_inn = seg_df["inn"].nunique()
    pct = len(default_inns) / total_inn * 100 if total_inn > 0 else 0
    
    print(f"{seg:<8} {total_inn:<12,} {len(has_trigger):<14,} {len(has_payment):<13,} {len(default_inns):<10,} {pct:.2f}%")
    
    results[seg] = {
        "total": total_inn,
        "has_trigger": has_trigger,
        "has_payment": has_payment,
        "default": default_inns,
        "seg_df": seg_df,
    }

# Итого
total_all = sum(r["total"] for r in results.values())
total_def = sum(len(r["default"]) for r in results.values())
print(f"\nИТОГО дефолтных ИНН: {total_def:,} из {total_all:,} ({total_def/total_all*100:.2f}%)")

### 3.1 Детализация: ИНН с триггером, но без платежа (и наоборот)

Анализ «неполных» случаев — компании, у которых сработал только один из двух критериев.

In [ ]:
print("Детализация по неполным случаям:\n")
print(f"{'Сегмент':<8} {'Только триггер':<18} {'Только платёж':<18} {'Оба (дефолт)':<15} {'Ни одного'}")
print("=" * 75)

for seg in ["МкБ", "МСБ", "ККБ"]:
    r = results[seg]
    only_trigger = r["has_trigger"] - r["has_payment"]
    only_payment = r["has_payment"] - r["has_trigger"]
    neither = r["total"] - len(r["has_trigger"] | r["has_payment"])
    
    print(f"{seg:<8} {len(only_trigger):<18,} {len(only_payment):<18,} {len(r['default']):<15,} {neither:,}")

print("\n📌 'Только триггер' — компании с ухудшением качества, но без просрочки.")
print("   Они могут быть на грани дефолта (предупреждающий сигнал).")
print("📌 'Только платёж' — компании с просрочкой, но без зафиксированного триггера.")
print("   Возможно, триггер-фактор ещё не был выявлен или не относится к данному сегменту.")

## 4. Временной анализ: последовательность факторов

Ключевой вопрос: **какой фактор появляется первым — триггер или платёж?**

Если триггер-фактор появился за 2 года до платёжного, возможно, они не связаны. Анализируем:
- Временной разрыв (gap) между первым появлением триггера и первым появлением платежа
- Распределение gap'ов
- Долю случаев, где триггер был раньше / позже / в один день

In [ ]:
gap_data = []  # список словарей для создания итогового DataFrame

for seg in ["МкБ", "МСБ", "ККБ"]:
    rules = DEFAULT_RULES[seg]
    r = results[seg]
    seg_df = r["seg_df"]
    
    for inn in r["default"]:
        inn_df = seg_df[seg_df["inn"] == inn]
        
        # Самая ранняя дата триггер-фактора у данного ИНН
        trigger_dates = inn_df[inn_df["fp_num"].isin(rules["trigger"])]["dt"]
        first_trigger = trigger_dates.min()
        
        # Самая ранняя дата платёжного фактора
        payment_dates = inn_df[inn_df["fp_num"].isin(rules["payment"])]["dt"]
        first_payment = payment_dates.min()
        
        # Разница в днях: положительное значение = платёж позже триггера
        gap_days = (first_payment - first_trigger).days
        
        gap_data.append({
            "segment": seg,
            "inn": inn,
            "first_trigger_date": first_trigger,
            "first_payment_date": first_payment,
            "gap_days": gap_days,
            "trigger_first": gap_days > 0,   # триггер раньше платежа
            "same_day": gap_days == 0,        # в один день
            "payment_first": gap_days < 0,    # платёж раньше триггера
        })

df_gap = pd.DataFrame(gap_data)
print(f"Всего дефолтных ИНН для временного анализа: {len(df_gap):,}")

In [ ]:
# ── Статистика по порядку появления факторов ──
print("Порядок появления факторов у дефолтных ИНН:\n")

for seg in ["МкБ", "МСБ", "ККБ"]:
    seg_gap = df_gap[df_gap["segment"] == seg]
    n = len(seg_gap)
    if n == 0:
        print(f"{seg}: нет дефолтных ИНН\n")
        continue
    
    n_trigger_first = seg_gap["trigger_first"].sum()
    n_same = seg_gap["same_day"].sum()
    n_payment_first = seg_gap["payment_first"].sum()
    
    print(f"{seg} ({n:,} ИНН):")
    print(f"  Триггер раньше платежа:  {n_trigger_first:>6,}  ({n_trigger_first/n*100:5.1f}%)")
    print(f"  В один день:             {n_same:>6,}  ({n_same/n*100:5.1f}%)")
    print(f"  Платёж раньше триггера:  {n_payment_first:>6,}  ({n_payment_first/n*100:5.1f}%)")
    print(f"  Медианный разрыв:        {seg_gap['gap_days'].median():.0f} дней")
    print(f"  Средний разрыв:          {seg_gap['gap_days'].mean():.0f} дней")
    print(f"  Мин:                     {seg_gap['gap_days'].min()} дней")
    print(f"  Макс:                    {seg_gap['gap_days'].max()} дней")
    print()

In [ ]:
# ── Гистограмма распределения временного разрыва ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, seg in enumerate(["МкБ", "МСБ", "ККБ"]):
    ax = axes[idx]
    seg_gap = df_gap[df_gap["segment"] == seg]["gap_days"]
    
    if len(seg_gap) == 0:
        ax.set_title(f"{seg}: нет данных")
        continue
    
    ax.hist(seg_gap, bins=50, color="#4472C4", edgecolor="white", alpha=0.8)
    ax.axvline(x=0, color="red", linestyle="--", linewidth=1.5, label="Один день")
    ax.axvline(x=seg_gap.median(), color="orange", linestyle="-", linewidth=1.5,
               label=f"Медиана ({seg_gap.median():.0f} дн.)")
    ax.set_title(f"{seg}: разрыв между триггером и платежом", fontsize=11, fontweight="bold")
    ax.set_xlabel("Дни (положит. = триггер раньше)")
    ax.set_ylabel("Количество ИНН")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Анализ с временным окном

Проверяем, как изменится количество дефолтов при ограничении **максимального разрыва** между факторами. Например, если требовать, чтобы оба фактора появились в пределах 6, 12 или 18 месяцев друг от друга.

In [ ]:
# Проверяем различные временные окна (в днях)
windows = [
    ("Без ограничения", None),
    ("± 90 дней (3 мес.)", 90),
    ("± 180 дней (6 мес.)", 180),
    ("± 365 дней (12 мес.)", 365),
    ("± 548 дней (18 мес.)", 548),
    ("± 730 дней (24 мес.)", 730),
]

print("Количество дефолтных ИНН при разных временных окнах:\n")
header = f"{'Окно':<28}"
for seg in ["МкБ", "МСБ", "ККБ"]:
    header += f" {seg:>10}"
header += f" {'ИТОГО':>10}"
print(header)
print("=" * 70)

for label, max_gap in windows:
    row = f"{label:<28}"
    total = 0
    for seg in ["МкБ", "МСБ", "ККБ"]:
        seg_gap = df_gap[df_gap["segment"] == seg]
        if max_gap is None:
            cnt = len(seg_gap)
        else:
            cnt = len(seg_gap[seg_gap["gap_days"].abs() <= max_gap])
        row += f" {cnt:>10,}"
        total += cnt
    row += f" {total:>10,}"
    print(row)

print("\n📌 Чем уже окно — тем строже критерий дефолта.")
print("   Слишком широкое окно (или его отсутствие) может приводить к ложным дефолтам,")
print("   когда факторы не связаны друг с другом по смыслу.")

## 6. Анализ порядка: триггер → платёж vs платёж → триггер

С точки зрения бизнес-логики ожидается, что **сначала** у компании ухудшается качество (триггер), а **затем** появляется просрочка (платёж). Обратный порядок может свидетельствовать о другой природе проблемы.

In [ ]:
# Дефолты только с «правильным» порядком: триггер раньше или в тот же день
print("Дефолты при условии: триггер ≤ платёж (триггер появился первым или в тот же день):\n")

print(f"{'Сегмент':<8} {'Все дефолты':<14} {'Триггер ≤ платёж':<18} {'Доля'}")
print("=" * 50)

for seg in ["МкБ", "МСБ", "ККБ"]:
    seg_gap = df_gap[df_gap["segment"] == seg]
    n_all = len(seg_gap)
    n_correct_order = len(seg_gap[seg_gap["gap_days"] >= 0])
    pct = n_correct_order / n_all * 100 if n_all > 0 else 0
    print(f"{seg:<8} {n_all:<14,} {n_correct_order:<18,} {pct:.1f}%")

print("\n📌 Если доля 'правильного порядка' низкая, это повод обсудить")
print("   с бизнесом, важен ли порядок появления факторов.")

## 7. Помесячная динамика: новые дефолты по месяцам

Дата дефолта = **максимальная** из двух дат (триггера и платежа), т.к. дефолт наступает, когда появляется **второй** (завершающий) фактор.

In [ ]:
# Дата дефолта = дата появления более позднего из двух факторов
df_gap["default_date"] = df_gap[["first_trigger_date", "first_payment_date"]].max(axis=1)
df_gap["default_month"] = df_gap["default_date"].dt.to_period("M")

# Помесячная динамика по сегментам
pivot = df_gap.groupby(["default_month", "segment"]).size().unstack(fill_value=0)
seg_order = [s for s in ["МкБ", "МСБ", "ККБ"] if s in pivot.columns]
pivot = pivot[seg_order]
pivot["Итого"] = pivot[seg_order].sum(axis=1)
pivot.index.name = "Месяц"

print("Помесячная динамика новых дефолтов:\n")
display(pivot)

# График
fig, ax = plt.subplots(figsize=(14, 5))
colors = {"МкБ": "#4472C4", "МСБ": "#ED7D31", "ККБ": "#70AD47"}
for seg in seg_order:
    ax.plot(pivot.index.astype(str), pivot[seg], marker="o", markersize=4,
            label=seg, color=colors.get(seg, "gray"), linewidth=1.5)

ax.set_title("Динамика новых дефолтов по месяцам", fontsize=13, fontweight="bold")
ax.set_ylabel("Количество ИНН")
ax.legend()
ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
plt.show()

## 8. Перекрёстная проверка: дефолтные ИНН и их факторы

Для дефолтных компаний показываем **полный набор факторов**, чтобы оценить, насколько «богат» профиль риска у этих компаний.

In [ ]:
for seg in ["МкБ", "МСБ", "ККБ"]:
    r = results[seg]
    if len(r["default"]) == 0:
        print(f"{seg}: нет дефолтных ИНН\n")
        continue
    
    seg_df = r["seg_df"]
    default_df = seg_df[seg_df["inn"].isin(r["default"])]
    
    # Среднее количество уникальных факторов у дефолтных vs недефолтных
    fp_per_inn_def = default_df.groupby("inn")["fp_num"].nunique()
    non_default_df = seg_df[~seg_df["inn"].isin(r["default"])]
    fp_per_inn_nondef = non_default_df.groupby("inn")["fp_num"].nunique()
    
    print(f"{'='*60}")
    print(f"{seg}: Среднее количество уникальных факторов на ИНН")
    print(f"  Дефолтные:      {fp_per_inn_def.mean():.1f} (медиана: {fp_per_inn_def.median():.0f})")
    print(f"  Не дефолтные:   {fp_per_inn_nondef.mean():.1f} (медиана: {fp_per_inn_nondef.median():.0f})")
    print()
    
    # Топ-10 самых частых факторов у дефолтных компаний
    top_fp = default_df["fp_num"].value_counts().head(10)
    print(f"{seg}: Топ-10 факторов у дефолтных компаний:")
    for fp, cnt in top_fp.items():
        print(f"  {fp:<12} {cnt:>6,}")
    print()

## 9. Примеры дефолтных компаний (выборка)

Для ручной верификации выведем несколько примеров с хронологией факторов.

In [ ]:
N_EXAMPLES = 3  # количество примеров на сегмент

for seg in ["МкБ", "МСБ", "ККБ"]:
    r = results[seg]
    rules = DEFAULT_RULES[seg]
    all_rule_factors = rules["trigger"] | rules["payment"]
    
    default_list = sorted(r["default"])
    sample = default_list[:N_EXAMPLES]
    
    if not sample:
        print(f"\n{seg}: нет дефолтных ИНН для примеров.")
        continue
    
    print(f"\n{'='*80}")
    print(f"{seg}: примеры дефолтных компаний (первые {N_EXAMPLES})")
    print(f"Правило: триггер {rules['trigger']} + платёж {rules['payment']}")
    print(f"{'='*80}")
    
    for inn in sample:
        inn_df = r["seg_df"][r["seg_df"]["inn"] == inn].sort_values("dt")
        
        # Показываем только записи с факторами из правил дефолта
        relevant = inn_df[inn_df["fp_num"].isin(all_rule_factors)][
            ["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE", "dt"]
        ].copy()
        relevant["роль"] = relevant["fp_num"].apply(
            lambda x: "ТРИГГЕР" if x in rules["trigger"] else "ПЛАТЁЖ"
        )
        
        gap_row = df_gap[(df_gap["inn"] == inn) & (df_gap["segment"] == seg)].iloc[0]
        
        print(f"\n  ИНН: {inn}")
        print(f"  Первый триггер: {gap_row['first_trigger_date']:%d.%m.%Y}")
        print(f"  Первый платёж:  {gap_row['first_payment_date']:%d.%m.%Y}")
        print(f"  Разрыв:         {gap_row['gap_days']} дней")
        print(f"  Всего факторов у ИНН: {len(inn_df)}")
        print(f"  Релевантные записи:")
        for _, row in relevant.iterrows():
            print(f"    {row['IDENTIFICATION_DATE']}  {row['fp_num']:<10}  {row['роль']:<8}  ({row['TYPE_FP']})")
        print()

## 10. Сводка и рекомендации

Итоговая таблица с ключевыми метриками для обсуждения с бизнесом.

In [ ]:
print("╔" + "═"*78 + "╗")
print("║" + " СВОДКА ПО РЕЗУЛЬТАТАМ АНАЛИЗА ЛОГИКИ ДЕФОЛТА".center(78) + "║")
print("╠" + "═"*78 + "╣")

for seg in ["МкБ", "МСБ", "ККБ"]:
    r = results[seg]
    rules = DEFAULT_RULES[seg]
    seg_gap = df_gap[df_gap["segment"] == seg]
    n_def = len(r["default"])
    pct = n_def / r["total"] * 100 if r["total"] > 0 else 0
    
    print(f"║ {seg}:".ljust(79) + "║")
    print(f"║   Правило: {sorted(rules['trigger'])} + {sorted(rules['payment'])}".ljust(79) + "║")
    print(f"║   Всего ИНН: {r['total']:,}  |  Дефолтных: {n_def:,} ({pct:.1f}%)".ljust(79) + "║")
    
    if len(seg_gap) > 0:
        med = seg_gap["gap_days"].median()
        correct = len(seg_gap[seg_gap["gap_days"] >= 0])
        pct_order = correct / len(seg_gap) * 100
        print(f"║   Медианный разрыв: {med:.0f} дн.  |  Триггер раньше: {pct_order:.0f}%".ljust(79) + "║")
    
    print("║" + " "*78 + "║")

print("╠" + "═"*78 + "╣")
print("║ РЕКОМЕНДАЦИИ:".ljust(79) + "║")
print("║".ljust(79) + "║")
print("║ 1. Согласовать с бизнесом МАКСИМАЛЬНОЕ ВРЕМЕННОЕ ОКНО между".ljust(79) + "║")
print("║    триггером и платежом (рекомендация: 12 мес.)".ljust(79) + "║")
print("║".ljust(79) + "║")
print("║ 2. Уточнить, важен ли ПОРЯДОК факторов (триггер → платёж).".ljust(79) + "║")
print("║".ljust(79) + "║")
print("║ 3. Проверить, не дублируются ли факторы из-за МИГРАЦИИ".ljust(79) + "║")
print("║    компаний между сегментами.".ljust(79) + "║")
print("║".ljust(79) + "║")
print("║ 4. Сравнить результат с текущей (действующей) логикой дефолта.".ljust(79) + "║")
print("║".ljust(79) + "║")
print("║ 5. Рассмотреть добавление ТРЕТЬЕГО фактора (например, СФП)".ljust(79) + "║")
print("║    для повышения точности классификации.".ljust(79) + "║")
print("╚" + "═"*78 + "╝")